#  Мониторинг процесса обучения

__Автор задач: Блохин Н.В. (NVBlokhin@fa.ru)__

Материалы: 
* Deep Learning with PyTorch (2020) Авторы: Eli Stevens, Luca Antiga, Thomas Viehmann 
* https://pytorch.org/tutorials/recipes/recipes/tensorboard_with_pytorch.html
* https://docs.wandb.ai/quickstart
* https://docs.wandb.ai/guides/track/log/log-summary#docusaurus_skipToContent_fallback
* https://docs.wandb.ai/guides/track/log/log-models
* https://www.youtube.com/playlist?list=PLD80i8An1OEGajeVo15ohAQYF1Ttle0lk

## Задачи для совместного разбора

1\. Рассмотрите возможности пакета `wandb` по отслеживанию числовых значений, визуализации изображений и таблиц.

## Задачи для самостоятельного решения

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import wandb
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

<p class="task" id="1"></p>

1\. Решите задачу регрессии, используя для мониторинга процесса обучения `wandb`. 

Разделите набор данных на обучающее и тестовое множество. В процессе обучения отслеживайте динамику изменения значения функции потерь и метрики $R^2$ по эпохам. После завершения обучения рассчитайте значение метрик MSE, RMSE, MAE и MAPE и сохраните в виде summary данного запуска. 

Обучите не менее трех моделей (с разной архитектурой или гиперпараметрами), отследите все запуски при помощи `wandb` и вставьте в текстовую ячейку скриншоты, демонстрирующие интерфейс `wandb` с графиками обучения. Для каждого запуска приложите также скриншот с описанием гиперпараметров модели и summary (страница overview).

- [ ] Проверено на семинаре

In [4]:
import torch 

X = torch.linspace(0, 1, 100).view(-1, 1)
y = torch.sin(2 * torch.pi * X) + 0.1 * torch.rand(X.size()) 

train_size = int(0.8 * len(X))
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]
X_train.shape, X_test.shape, y_train.shape, y_test.shape

(torch.Size([80, 1]),
 torch.Size([20, 1]),
 torch.Size([80, 1]),
 torch.Size([20, 1]))

In [5]:
class Model1(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )

    def forward(self, x):
        return self.net(x)


class Model2(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 32),
            nn.Tanh(),
            nn.Linear(32, 32),
            nn.Tanh(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.net(x)


class Model3(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.net(x)

In [8]:
def compute_metrics(y_true, y_pred):
    y_true = y_true.detach().numpy()
    y_pred = y_pred.detach().numpy()

    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

    return mse, rmse, mae, mape

In [15]:
def train_model(model_class, config):
    wandb.init(project="22", config=config)

    model = model_class()
    optimizer = optim.Adam(model.parameters(), lr=config["lr"])
    criterion = nn.MSELoss()

    for epoch in range(config["epochs"]):
        model.train()

        preds = model(X_train)
        loss = criterion(preds, y_train)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # eval
        model.eval()
        with torch.no_grad():
            test_preds = model(X_test)
            test_loss = criterion(test_preds, y_test)

            mse, rmse, mae, mape = compute_metrics(y_test, test_preds)

        # логирование
        wandb.log({
            "epoch": epoch,
            "train_loss": loss.item(),
            "test_loss": test_loss.item(),
            "MSE": mse,
            "RMSE": rmse,
            "MAE": mae,
            "MAPE": mape
        })

    # финальные метрики в summary
    wandb.summary["final_MSE"] = mse
    wandb.summary["final_RMSE"] = rmse
    wandb.summary["final_MAE"] = mae
    wandb.summary["final_MAPE"] = mape

    wandb.finish()

In [19]:
train_model(Model1, {"lr": 0.01, "epochs": 30})

MAE,▁▁▂▃▃▄▅▆▇▇████▇▇▆▆▅▅▄▄▃▃▃▂▂▂▂▂
MAPE,█▇▅▄▃▂▁▁▂▃▃▃▃▃▂▂▁▁▂▂▃▃▄▅▅▆▇▇██
MSE,▁▁▂▃▃▄▅▆▇▇████▇▇▆▆▅▅▄▄▃▃▂▂▂▂▂▁
RMSE,▁▁▂▃▄▅▆▆▇██████▇▇▆▆▅▅▄▄▃▃▃▂▂▂▂
epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
test_loss,▁▁▂▃▃▄▅▆▇▇████▇▇▆▆▅▅▄▄▃▃▂▂▂▂▂▁
train_loss,█▇▆▅▄▄▄▃▃▃▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁
MAE,0.30936
MAPE,1264.63025
MSE,0.12605
RMSE,0.35503


In [17]:
train_model(Model2, {"lr": 0.005, "epochs": 150})

MAE,▃▂▁▁▂▅▆▆▅▄▃▃▄▄▄▅▅▅▅▅▄▅▅▅▆▆▆▆▆▇█████▆▅▅▅▄
MAPE,▁▁▃▆▇▇▇▇▆▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇▇███████▇▇▇▇▆▆▆▅
MSE,▃▂▁▁▆▄▃▂▂▃▃▃▃▄▄▄▄▄▄▄▄▅▅▅▇▇█████▇▇▆▆▅▅▄▄▃
RMSE,▁▁▁▂▃▆▆▆▅▄▃▃▅▅▅▅▅▅▆▆▆▇▇▇▇▇██████▇▇▆▅▅▅▄▄
epoch,▁▁▁▁▂▂▂▂▂▂▂▂▃▃▃▃▃▃▃▃▄▄▄▄▄▄▄▄▅▅▅▅▅▅▆▇▇███
test_loss,▂▂▁▂▃▆▆▆▅▄▃▂▂▃▄▄▄▄▅▅▆▆▆▆▇█████▆▆▆▆▅▄▄▄▄▃
train_loss,█▇▄▅▄▄▄▄▄▄▄▄▄▄▄▄▃▃▃▃▃▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▁▁▁▁
MAE,0.59871
MAPE,3455.07397
MSE,0.45743
RMSE,0.67634


In [21]:
train_model(Model3, {"lr": 0.001, "epochs": 200})

MAE,▁▁▁▁▁▃▃▃▃▄▄▅▅▅▅▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇██████████
MAPE,▁▁▁▁▁▂▃▃▄▄▅▅▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇█████████████
MSE,▂▂▂▂▂▁▁▁▁▁▂▂▂▃▃▃▃▄▄▄▄▅▅▅▆▆▆▇▇▇▇▇▇▇▇▇████
RMSE,▃▃▃▃▂▁▁▂▃▃▄▄▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇██████████
epoch,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇████
test_loss,▂▂▂▁▁▁▁▂▂▃▄▄▄▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇████████████
train_loss,█▇▇▆▆▅▅▅▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁
MAE,1.05791
MAPE,5779.06738
MSE,1.36331
RMSE,1.16761


<p class="task" id="2"></p>

2\. Решите задачу классификации, используя для мониторинга процесса обучения `wandb`. 

Разделите набор данных на обучающее и тестовое множество. В процессе обучения отслеживайте динамику изменения значения функции потерь и метрики `Accuracy` по эпохам. После завершения обучения рассчитайте значение метрик Accuracy, Precision, Recall и F1 и сохраните в виде summary данного запуска. 

Отследите все запуски при помощи `wandb` и вставьте в текстовую ячейку скриншоты, демонстрирующие интерфейс `wandb` с графиками обучения. Для каждого запуска приложите также скриншот с описанием гиперпараметров модели и summary (страница overview).


- [ ] Проверено на семинаре

In [22]:
from sklearn.datasets import make_circles

X, y = make_circles(n_samples=1000, noise=0.05, random_state=42)
X = th.FloatTensor(X)
y = th.LongTensor(y)

NameError: name 'th' is not defined

<p class="task" id="3"></p>

3\. Повторите задачу 2, вычислив и визуализировав матрицу несоответствий (для обучающей и тестовой выборки) тремя способами при помощи `wandb`:
* используя `torchmetrics` и представив данные в виде объекта `wandb.Table`;
* используя готовую функцию `wandb.plot.confusion_matrix`;
* построив тепловую карту при помощи `seaborn` и представив данные в виде объекта `wandb.Image`.

Вставьте в текстовую ячейку скриншоты, демонстрирующие интерфейс `wandb` со всеми нужными визуализациями.


- [ ] Проверено на семинаре

<p class="task" id="4"></p>

4\. Повторите задачу 2, обучив две модели: линейную и нелинейную. Для каждой из моделей сделайте прогноз (по всей выборке) и визуализируйте облако точек в виде `wandb.Image` (раскрасьте точки в цвета, соответствующие прогнозам модели).

Вставьте в текстовую ячейку скриншоты, демонстрирующие интерфейс `wandb` со всеми нужными визуализациями.


- [ ] Проверено на семинаре

<p class="task" id="5"></p>

5\. Повторите задачу 2, реализовав логику ранней остановки. Для этого разделите данные на три части: обучающую, валидационную и тестовую. Остановите процесс обучения, если целевая метрика (F1) на валидации не увеличивалась в течении последних $k$ ($k$ - гиперпараметр метода) эпох. В момент остановки выведите сообщение с текущим номером эпохи. Сохраните номер эпохи, на которой процесс обучения был прерван, в виде summary данного запуска.

Помимо отслеживания метрик на обучающей и тестовой выборке, также отслеживайте метрики на валидационной выборке в процессе обучения. 

Постройте таблицу `wandb.Table`, в которой содержится информация о:
* признаках объекта;
* правильном ответе;
* прогнозе модели;
* принадлежности к обучающему, валидационному или тестовому множеству.

Визуализируйте данную таблицу при помощи `wandb`.

Вставьте в текстовую ячейку скриншоты, демонстрирующие интерфейс `wandb` со всеми нужными визуализациями.

- [ ] Проверено на семинаре
